# ⚡ Kaggle Dual T4 LLM Server (Ollama) + OpenAI `/v1`
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://github.com/SudhirDevOps1/t4-gpu-llm-api/blob/main/kaggle/01_dual_t4_llm_engine.ipynb)
[![Notebook Store](https://img.shields.io/badge/Notebook_Store-Kaggle_Dual_T4-20BEFF?style=for-the-badge&logo=kaggle)](https://github.com/SudhirDevOps1/t4-gpu-llm-api)

> **Honest hardware note:** Kaggle gives you 2× Tesla T4 (30 GB total), but Ollama does **layer
> splitting**, not tensor parallelism. A 14B Q4 model (~9 GB) fits comfortably on a **single** T4, so the
> second GPU mostly stays idle. For true tensor-parallel serving you need vLLM with a model that does not
> fit on one card.

- **Security:** Ollama ignores the `Authorization` header, so this notebook puts a **Basic-auth reverse
  proxy** on `:11435` and tunnels only that port. Ollama itself stays on `127.0.0.1:11434`.
- **API key:** use the generated session password as the bearer token in any OpenAI-compatible client.
- **Quota:** 9 h/session, 30 h/week on the free tier.


In [ ]:
#@title 🚀 1. Hardware Diagnostic
import hashlib, os, socket, time, uuid

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

import torch
count = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f"🔥 Detected GPUs: {count}")
for i in range(count):
    print(f"  [GPU {i}] {torch.cuda.get_device_name(i)} — "
          f"{torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GB")

raw_fp = f"kaggle-dual-{socket.gethostname()}-{uuid.uuid4()}-{time.time()}"
INSTANCE_ID = "kaggle-dual-" + hashlib.sha256(raw_fp.encode()).hexdigest()[:8]
print(f"🆔 Unique Session Fingerprint: {INSTANCE_ID}")


In [ ]:
#@title 📦 2. Install Ollama, cloudflared & aiohttp
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q aiohttp
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("✅ Ollama + cloudflared + aiohttp installed.")


In [ ]:
#@title ⚡ 3. Start Ollama on localhost only
import os
import socket
import subprocess
import time

subprocess.run(["pkill", "-9", "ollama"], check=False)
time.sleep(1)

server_env = {
    **os.environ,
    "OLLAMA_HOST": "127.0.0.1:11434",   # never bind the tunnel directly to this
    "OLLAMA_NUM_PARALLEL": "1",
    "OLLAMA_KEEP_ALIVE": "24h",
}
subprocess.Popen(["ollama", "serve"], env=server_env,
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):
    time.sleep(1)
    with socket.socket() as sock:
        if sock.connect_ex(("127.0.0.1", 11434)) == 0:
            print("✅ Ollama is listening on 127.0.0.1:11434")
            break
else:
    raise RuntimeError("Ollama did not start.")


In [ ]:
%%writefile auth_proxy.py
#!/usr/bin/env python3
"""Password-protected reverse proxy for notebook servers (Ollama / ComfyUI).

Colab and Kaggle tunnels expose a port to the public internet. Ollama ignores
the Authorization header entirely and ComfyUI ships without any auth layer, so
an open tunnel means anyone with the URL can use (or reconfigure) your runtime.

This proxy sits on a second port, requires HTTP Basic auth (or
``Authorization: Bearer <password>`` so OpenAI-compatible clients keep working),
and forwards HTTP + WebSocket traffic to the local service.

    python auth_proxy.py --target-port 8188 --port 8189
    PROXY_PASSWORD=... python auth_proxy.py --target-port 11434 --port 11435

Only aiohttp is required (already installed by ComfyUI; one-line pip otherwise).
"""

from __future__ import annotations

import argparse
import asyncio
import base64
import hmac
import os
import sys

from aiohttp import ClientSession, ClientTimeout, TCPConnector, WSMsgType, web

# Headers that must not be forwarded between the two hops.
SKIP_REQUEST = {
    "host", "connection", "keep-alive", "proxy-authenticate", "proxy-authorization",
    "te", "trailer", "transfer-encoding", "upgrade", "expect", "origin", "referer",
}
SKIP_RESPONSE = {
    "connection", "keep-alive", "proxy-authenticate", "proxy-authorization",
    "te", "trailer", "transfer-encoding", "upgrade", "content-encoding", "content-length",
}

SESSION = web.AppKey("session", ClientSession)


def build_app(target_port: int, user: str, password: str) -> web.Application:
    basic = "Basic " + base64.b64encode(f"{user}:{password}".encode()).decode()
    bearer = f"Bearer {password}"
    backend = f"127.0.0.1:{target_port}"

    async def start_session(app: web.Application) -> None:
        # Created on the running loop (required by aiohttp >= 3.10).
        app[SESSION] = ClientSession(
            connector=TCPConnector(limit=128),
            timeout=ClientTimeout(total=None, sock_connect=30, sock_read=None),
        )

    async def close_session(app: web.Application) -> None:
        await app[SESSION].close()

    def authorized(request: web.Request) -> bool:
        supplied = request.headers.get("Authorization", "")
        return hmac.compare_digest(supplied, basic) or hmac.compare_digest(supplied, bearer)

    def reject(request: web.Request) -> web.Response:
        return web.Response(
            status=401,
            text="401: authentication required\n",
            headers={"WWW-Authenticate": 'Basic realm="notebook", charset="UTF-8"'},
        )

    async def proxy_http(request: web.Request) -> web.StreamResponse:
        if not authorized(request):
            return reject(request)
        headers = {k: v for k, v in request.headers.items() if k.lower() not in SKIP_REQUEST}
        body = await request.read() if request.body_exists else None
        try:
            async with request.app[SESSION].request(
                request.method,
                f"http://{backend}{request.rel_url}",
                headers=headers,
                data=body,
                allow_redirects=False,
            ) as upstream:
                response = web.StreamResponse(status=upstream.status, reason=upstream.reason)
                for key, value in upstream.headers.items():
                    if key.lower() not in SKIP_RESPONSE:
                        response.headers[key] = value
                await response.prepare(request)
                if request.method != "HEAD":
                    async for chunk in upstream.content.iter_any():
                        await response.write(chunk)
                await response.write_eof()
                return response
        except Exception as exc:  # backend down / refused
            return web.Response(status=502, text=f"502: backend unavailable ({exc})\n")

    async def proxy_ws(request: web.Request) -> web.StreamResponse:
        if not authorized(request):
            return reject(request)
        client = web.WebSocketResponse(heartbeat=30)
        await client.prepare(request)
        try:
            async with request.app[SESSION].ws_connect(
                f"ws://{backend}{request.rel_url}", heartbeat=30, max_msg_size=0
            ) as upstream:

                async def pump(src, dst) -> None:
                    async for message in src:
                        if message.type == WSMsgType.TEXT:
                            await dst.send_str(message.data)
                        elif message.type == WSMsgType.BINARY:
                            await dst.send_bytes(message.data)
                        elif message.type in (WSMsgType.CLOSE, WSMsgType.CLOSING,
                                              WSMsgType.CLOSED, WSMsgType.ERROR):
                            break

                up = asyncio.create_task(pump(client, upstream))
                down = asyncio.create_task(pump(upstream, client))
                await asyncio.wait({up, down}, return_when=asyncio.FIRST_COMPLETED)
                for task in (up, down):
                    task.cancel()
        except Exception:
            pass
        finally:
            await client.close()
        return client

    app = web.Application(client_max_size=1024 ** 3)
    app.router.add_get("/ws", proxy_ws)
    app.router.add_route("*", "/{tail:.*}", proxy_http)
    app.on_startup.append(start_session)
    app.on_cleanup.append(close_session)
    return app


def main() -> int:
    parser = argparse.ArgumentParser(description="Basic-auth reverse proxy for notebook servers")
    parser.add_argument("--target-port", type=int, required=True, help="local service port (e.g. 8188)")
    parser.add_argument("--port", type=int, default=0, help="listen port (default: target + 1)")
    parser.add_argument("--host", default="127.0.0.1")
    parser.add_argument("--user", default=os.getenv("PROXY_USER", "admin"))
    parser.add_argument("--password", default=os.getenv("PROXY_PASSWORD", ""))
    args = parser.parse_args()

    if not args.password:
        print("error: set PROXY_PASSWORD or pass --password", file=sys.stderr)
        return 2

    port = args.port or args.target_port + 1
    print(f"auth proxy: 127.0.0.1:{port} -> 127.0.0.1:{args.target_port} (user={args.user})")
    web.run_app(build_app(args.target_port, args.user, args.password),
                host=args.host, port=port, print=None)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
#@title 🔐 4. Start the Auth Proxy
import os
import secrets
import socket
import subprocess
import sys
import time

if not os.environ.get("STUDIO_PASSWORD"):
    os.environ["STUDIO_PASSWORD"] = secrets.token_urlsafe(9)
PASSWORD = os.environ["STUDIO_PASSWORD"]

log = open("/tmp/auth_proxy.log", "w")
subprocess.Popen(
    [sys.executable, "auth_proxy.py", "--target-port", "11434", "--port", "11435"],
    env={**os.environ, "PROXY_PASSWORD": PASSWORD},
    stdout=log,
    stderr=subprocess.STDOUT,
)

for _ in range(30):
    time.sleep(0.5)
    with socket.socket() as sock:
        if sock.connect_ex(("127.0.0.1", 11435)) == 0:
            break
else:
    raise RuntimeError("Auth proxy did not start — check /tmp/auth_proxy.log")

print("✅ Auth proxy listening on 127.0.0.1:11435 (Basic auth user: admin)")
print(f"🔑 Password for this session: {PASSWORD}")


In [ ]:
#@title 📥 5. Pull a 14B Model
# Q4_K_M ≈ 9 GB: fits on one T4 with room for the KV cache.
!ollama pull deepseek-r1:14b
!ollama list


In [ ]:
#@title 🌐 6. Expose the Auth Proxy (OpenAI-compatible /v1)
import os
import re
import subprocess
import time
from IPython.display import Markdown, clear_output, display

LOG_PATH = "/tmp/cloudflared.log"
if os.path.exists(LOG_PATH):
    os.remove(LOG_PATH)
subprocess.run(["pkill", "-9", "cloudflared"], check=False)
time.sleep(1)

log_file = open(LOG_PATH, "w")
subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:11435"],
                 stdout=log_file, stderr=subprocess.STDOUT)

public_url = None
for _ in range(40):
    time.sleep(1)
    if os.path.exists(LOG_PATH):
        match = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", open(LOG_PATH).read())
        if match:
            public_url = match.group(0)
            break

clear_output()
if not public_url:
    raise RuntimeError("Tunnel failed — check /tmp/cloudflared.log")

display(Markdown(f"""
### 🚀 API is online — password protected

| Setting | Value |
| :--- | :--- |
| **OpenAI Base URL** | `{public_url}/v1` |
| **API key** | `{PASSWORD}` (sent as `Authorization: Bearer ...`) |
| **Model** | `deepseek-r1:14b` |
| **Direct backend** | `127.0.0.1:11434` (not exposed) |

```bash
curl {public_url}/v1/chat/completions \\
  -H "Content-Type: application/json" \\
  -H "Authorization: Bearer {PASSWORD}" \\
  -d '{{"model": "deepseek-r1:14b", "messages": [{{"role": "user", "content": "Hello"}}]}}'
```
"""))
